# Question 3 / Phase 3: Shortest Path and Route Planning

This notebook contains the third question/phase of the Mini-Courier Planner capstone assignment.

## Theory

Phase 3 uses graph algorithms for route planning.

Dijkstra's algorithm finds the shortest distance from one source node to all other nodes in a weighted graph with non-negative edge weights.

The nearest-neighbor heuristic then builds a practical route:

1. Start at the warehouse.
2. Visit the closest unvisited selected destination.
3. Repeat until all selected destinations are visited.
4. Return to the warehouse.

This route is approximate, not guaranteed globally optimal.

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple


@dataclass(frozen=True)
class Parcel:
    parcel_id: str
    destination: str
    weight: int
    value: int
    deadline: int
    priority: str


PRIORITY_SCORE = {
    "High": 3,
    "Medium": 2,
    "Low": 1,
}


def priority_score(parcel: Parcel) -> int:
    return PRIORITY_SCORE[parcel.priority]


def parcel_benefit(parcel: Parcel) -> int:
    urgency_bonus = max(0, 6 - parcel.deadline) * 5
    priority_bonus = priority_score(parcel) * 20
    return parcel.value + priority_bonus + urgency_bonus


parcels = [
    Parcel("P01", "A", 4, 60, 2, "High"),
    Parcel("P02", "B", 2, 35, 1, "High"),
    Parcel("P03", "C", 6, 80, 4, "Medium"),
    Parcel("P04", "D", 3, 45, 3, "High"),
    Parcel("P05", "E", 5, 70, 2, "Medium"),
    Parcel("P06", "F", 1, 20, 1, "Low"),
    Parcel("P07", "G", 7, 100, 5, "High"),
    Parcel("P08", "H", 2, 25, 3, "Low"),
    Parcel("P09", "I", 2, 30, 3, "High"),
]

VEHICLE_CAPACITY = 12

In [ ]:
def sort_parcels_by_priority_and_deadline(parcel_list: List[Parcel]) -> List[Parcel]:
    return sorted(
        parcel_list,
        key=lambda parcel: (-priority_score(parcel), parcel.deadline),
    )


def knapsack_select_parcels(
    parcel_list: List[Parcel], capacity: int
) -> Tuple[int, List[Parcel], List[List[int]]]:
    n = len(parcel_list)
    dp = [[0 for _ in range(capacity + 1)] for _ in range(n + 1)]

    for i in range(1, n + 1):
        parcel = parcel_list[i - 1]
        weight = parcel.weight
        benefit = parcel_benefit(parcel)

        for current_capacity in range(capacity + 1):
            without_parcel = dp[i - 1][current_capacity]
            with_parcel = 0

            if weight <= current_capacity:
                with_parcel = benefit + dp[i - 1][current_capacity - weight]

            dp[i][current_capacity] = max(without_parcel, with_parcel)

    selected = []
    remaining_capacity = capacity

    for i in range(n, 0, -1):
        if dp[i][remaining_capacity] != dp[i - 1][remaining_capacity]:
            parcel = parcel_list[i - 1]
            selected.append(parcel)
            remaining_capacity -= parcel.weight

    selected.reverse()
    return dp[n][capacity], selected, dp


sorted_parcels = sort_parcels_by_priority_and_deadline(parcels)
maximum_benefit, selected_parcels, dp_table = knapsack_select_parcels(
    sorted_parcels,
    VEHICLE_CAPACITY,
)

total_weight = sum(parcel.weight for parcel in selected_parcels)
total_benefit = sum(parcel_benefit(parcel) for parcel in selected_parcels)
remaining_capacity = VEHICLE_CAPACITY - total_weight

In [ ]:
from heapq import heappop, heappush
from typing import Dict


Graph = Dict[str, Dict[str, int]]


road_graph: Graph = {
    "Warehouse": {"A": 4, "B": 2, "C": 7},
    "A": {"Warehouse": 4, "B": 1, "D": 5, "E": 8},
    "B": {"Warehouse": 2, "A": 1, "C": 3, "D": 4, "F": 6},
    "C": {"Warehouse": 7, "B": 3, "E": 2, "G": 5},
    "D": {"A": 5, "B": 4, "F": 2, "I": 3},
    "E": {"A": 8, "C": 2, "G": 3, "H": 4},
    "F": {"B": 6, "D": 2, "H": 3, "I": 4},
    "G": {"C": 5, "E": 3, "H": 2},
    "H": {"E": 4, "F": 3, "G": 2, "I": 2},
    "I": {"D": 3, "F": 4, "H": 2},
}


def dijkstra(graph: Graph, start: str) -> Tuple[Dict[str, int], Dict[str, str]]:
    distances = {node: float("inf") for node in graph}
    previous = {node: None for node in graph}
    distances[start] = 0
    priority_queue = [(0, start)]

    while priority_queue:
        current_distance, current_node = heappop(priority_queue)

        if current_distance > distances[current_node]:
            continue

        for neighbor, edge_weight in graph[current_node].items():
            new_distance = current_distance + edge_weight

            if new_distance < distances[neighbor]:
                distances[neighbor] = new_distance
                previous[neighbor] = current_node
                heappush(priority_queue, (new_distance, neighbor))

    return distances, previous


def shortest_path(previous: Dict[str, str], destination: str) -> List[str]:
    path = []
    current = destination

    while current is not None:
        path.append(current)
        current = previous[current]

    return list(reversed(path))

In [ ]:
selected_destinations = [parcel.destination for parcel in selected_parcels]
warehouse_distances, warehouse_previous = dijkstra(road_graph, "Warehouse")

print("Selected parcels:", ", ".join(parcel.parcel_id for parcel in selected_parcels))
print("Selected destinations:", ", ".join(selected_destinations))
print()
print("Shortest distance from warehouse to selected destinations:")

for destination in selected_destinations:
    path = shortest_path(warehouse_previous, destination)
    print(
        f"{destination}: distance={warehouse_distances[destination]} | "
        f"path={' -> '.join(path)}"
    )

In [ ]:
def plan_nearest_neighbor_route(
    graph: Graph, start: str, destinations: List[str]
) -> Tuple[List[str], int]:
    route = [start]
    total_distance = 0
    current = start
    unvisited = set(destinations)

    while unvisited:
        distances, _ = dijkstra(graph, current)
        next_destination = min(unvisited, key=lambda destination: distances[destination])
        total_distance += distances[next_destination]
        route.append(next_destination)
        current = next_destination
        unvisited.remove(next_destination)

    distances, _ = dijkstra(graph, current)
    total_distance += distances[start]
    route.append(start)

    return route, total_distance


delivery_route, route_distance = plan_nearest_neighbor_route(
    road_graph,
    "Warehouse",
    selected_destinations,
)

print("Nearest-neighbor delivery route:", " -> ".join(delivery_route))
print(f"Approximate total route distance: {route_distance}")

## Phase 3 Summary

Phase 3 converts selected parcel destinations into a graph problem. Dijkstra's algorithm calculates shortest paths, and the nearest-neighbor heuristic creates a practical delivery route from the warehouse and back.